In [1]:

import torch, os
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig, set_seed


model_name = "meta-llama/Llama-3.2-3B-Instruct"

/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [3]:
from datasets import load_from_disk
dataset_based_path = "/home/jovyan/Works/MSc_Thesis/dataset"
dataset = load_from_disk(f"{dataset_based_path}/mh-mental-chat-16k_v1")

In [4]:
dataset["train"][0]

{'instruction': "You are a helpful mental health counselling assistant, please answer the mental health questions based on the patient's description. \nThe assistant gives helpful, comprehensive, and appropriate answers to the user's questions. ",
 'query': "Anxiety makes it difficult for me to trust others. I constantly worry about being judged or rejected, so I tend to isolate myself. It's a vicious cycle because I crave connection, but my anxiety prevents me from fully engaging with others.",
 'response': "It can be challenging when anxiety interferes with your ability to trust others and form connections. It's understandable that worrying about being judged or rejected can lead you to isolate yourself as a way of protecting yourself from potential pain or disappointment. However, it's important to remember that isolation can further perpetuate the cycle of anxiety and make it even more difficult to engage with others.\n\nOne suggestion I have is to start small and gradually expose 

In [5]:
# Load the model with full precisoin
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,  # Full precision
).to(device)

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 42.26it/s]


In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [7]:
from peft import LoraConfig, TaskType

rank_dimension = 16  # rank dimension for LoRA update matrices (smaller = more compression)
lora_alpha = 64      # lora_alpha: scaling factor for LoRA layers (higher = stronger adaptation)
lora_dropout = 0.05   # lora_dropout: dropout probability for LoRA layers (helps prevent overfitting)

peft_config = LoraConfig(
    r=rank_dimension,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention layers (most important)
        "gate_proj", "up_proj", "down_proj",     # MLP layers
        "embed_tokens", "lm_head"                # Need for stronger adaptation
    ],
    task_type=TaskType.CAUSAL_LM,
    # bias="none"  # Add this for better performance
)

In [8]:
from trl import SFTTrainer, SFTConfig

# Configure training arguments for mental health function calling
training_args = SFTConfig(
    output_dir="/home/jovyan/Works/MSc_Thesis/mh_conversation/finetuned_model/llama3.2_3b_mh_conversation_v3",
    
    # Batch configuration - small batches with accumulation for memory efficiency
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,  # Effective batch size = 4
    
    # Learning configuration optimized for function calling
    learning_rate=1e-4,  # Conservative rate for stable training
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
    weight_decay=0.1,
    
    # Training duration - more epochs for smaller specialized dataset
    num_train_epochs=4,
    
    # Evaluation and monitoring
    eval_strategy="epoch",                    # Evaluate every epoch
    save_strategy="epoch",                    # Save every epoch (changed from "no")
    save_total_limit=3,                       # Keep only 3 best checkpoints
    load_best_model_at_end=True,             # Load best model at end
    metric_for_best_model="eval_loss",       # Use validation loss as metric
    greater_is_better=False,                # Lower loss is better
    
    # Memory optimization for 3B model
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    
    # Sequence handling for mental health conversations
    max_seq_length=2000,  # Adequate for conversation
    packing=True,  # Efficient training on variable length sequences
    # dataset_text_field="text",
    
    push_to_hub=False,
)

print(f"📊 Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")


📊 Effective batch size: 8


In [9]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    peft_config=peft_config,
)

/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:574: UserWarning: Model with `tie_word_embeddings=True` and the tied_target_modules=['model.embed_tokens', 'lm_head'] are part of the adapter. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. See for example https://github.com/huggingface/peft/issues/2018.
  warnings.warn(
/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/trl/trainer/sft_trainer.py:412: UserWarning: Padding-free training is enabled, but the attention implementation is not set to 'flash_attention_2'. Padding-free training flattens batches into a single sequence, and 'flash_attention_2' is the only known attention mechanism that reliably supports this. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation='flash_attention_2'` in the model configuration, or verify that your att

In [10]:
print("🚀 Starting training...")
trainer.train()

🚀 Starting training...


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch,Training Loss,Validation Loss
1,0.865400,0.861925
2,0.807800,0.826799
3,0.757800,0.814306
4,0.731400,0.814734


/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/peft/utils/save_and_load.py:222: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/peft/utils/save_and_load.py:222: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `t

TrainOutput(global_step=868, training_loss=0.8235329883988551, metrics={'train_runtime': 5159.6483, 'train_samples_per_second': 2.683, 'train_steps_per_second': 0.168, 'total_flos': 4.676809052120924e+17, 'train_loss': 0.8235329883988551})

In [11]:
# Save the final model
print("Saving model...")
trainer.save_model()
tokenizer.save_pretrained(training_args.output_dir)

print("Training complete!")
print(f"Model saved to: {training_args.output_dir}")

Saving model...


/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/peft/utils/save_and_load.py:222: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


Training complete!
Model saved to: /home/jovyan/Works/MSc_Thesis/mh_conversation/finetuned_model/llama3.2_3b_mh_conversation_v3
